# Lizard Parameter Count — Raw Output Extraction (Java)

This notebook analyzes **Java repositories** with **Lizard** and captures complete raw tool output for Parameter Count, Cyclomatic Complexity (CCN), Function Length, NLOC, Token Count, and Maximum Nesting Depth.

**Default benchmark repository:** [spring-projects/spring-petclinic](https://github.com/spring-projects/spring-petclinic)

> **Note:** **Parameter Count is directly emitted by Lizard** through the `PARAM` column in CSV output. No derivation is required for this metric.


## Section 1 — Install Dependencies

Install open-source Python packages and verify Lizard.


In [ ]:
!pip install -q lizard pandas gitpython jupyter

import subprocess, sys
subprocess.run([sys.executable, '-m', 'lizard', '--version'], check=False)


## Section 2 — Configuration

Set repository source, workspace, and output directory.


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/spring-projects/spring-petclinic.git'

LOCAL_REPO_PATH = '/content/spring-petclinic'

WORKSPACE_DIR = './workspace'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'

CLONE_DEPTH = 1

RAW_OUTPUT_PREVIEW_LINES = 150

# Fast validation benchmark:
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/parameter_count_benchmark'


## Section 3 — Imports and Utility Functions


In [ ]:
from pathlib import Path

from __future__ import annotations

import csv
import io
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

EXCLUDED_DIR_NAMES = {
    ".git", "target", "build", "out", ".gradle", ".mvn", "generated-sources", "docs",
}
LIZARD_EXCLUDE_PATTERNS = [
    "*/.git/*", "*/target/*", "*/build/*", "*/out/*", "*/.gradle/*",
    "*/.mvn/*", "*/generated-sources/*", "*/docs/*",
]
LIZARD_RAW_COLUMNS = [
    "nloc", "ccn", "token_count", "parameter_count", "length", "location",
    "file", "function", "long_name", "start_line", "end_line",
]
LIZARD_OUTPUT_COLUMNS = [
    "NLOC", "CCN", "token", "PARAM", "length", "location", "file", "function",
]
LIZARD_METRICS_COLUMNS = [
    "file", "class", "method", "nloc", "cyclomatic_complexity", "token_count",
    "parameter_count", "function_length", "start_line", "end_line",
]
LONG_PARAMETER_LIST_COLUMNS = ["file", "class", "method", "parameter_count", "status"]
LONG_PARAMETER_THRESHOLD = 5
PY = sys.executable


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        if not self.error_log_path.exists():
            self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, file: str = "notebook") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append({"timestamp": timestamp, "file": file, "error_message": message})
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "file", "error_message"])
            writer.writeheader()
            writer.writerows(self._errors)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str, workspace_dir: Path, if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)
    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError("IF_CLONE_EXISTS must be 'reuse' or 'reclone'.")
    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}", file=repo_url)
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise NotADirectoryError(msg)
    try:
        Repo(local_repo_path)
        logger.info("Validated Git repository.")
    except InvalidGitRepositoryError:
        logger.info("Path is not a Git repository; proceeding as source directory.")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool, repo_url: str, local_repo_path: str | Path, workspace_dir: Path,
    if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo_path), logger)


def discover_java_files(repo_path: Path) -> list[Path]:
    files: list[Path] = []
    for path in repo_path.rglob("*.java"):
        if any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            continue
        files.append(path.resolve())
    return sorted(files)


def compute_repository_stats(repo_path: Path, java_files: list[Path]) -> dict[str, Any]:
    total_size = sum(path.stat().st_size for path in java_files)
    directories = {path.parent for path in java_files}
    return {
        "repository_name": repo_path.name,
        "repository_size_bytes": total_size,
        "directory_count": len(directories),
        "java_file_count": len(java_files),
    }


def save_java_inventory(java_files: list[Path], output_csv: Path) -> None:
    pd.DataFrame(
        [{"file_path": str(p), "file_name": p.name, "directory": str(p.parent)} for p in java_files]
    ).to_csv(output_csv, index=False)


def split_java_symbol(function_name: str) -> tuple[str, str]:
    name = str(function_name).strip('"')
    if "::" in name:
        class_name, method_name = name.split("::", 1)
        return class_name, method_name
    return "", name


def build_lizard_command(repo_path: Path, *, csv_output: bool = False, ens: bool = False) -> list[str]:
    command = [PY, "-m", "lizard", "-l", "java"]
    for pattern in LIZARD_EXCLUDE_PATTERNS:
        command.extend(["-x", pattern])
    if csv_output:
        command.append("--csv")
    if ens:
        command.append("-ENS")
    command.append(str(repo_path))
    return command


def run_lizard_command(command: list[str], logger: NotebookLogger) -> tuple[str, str, int]:
    completed = subprocess.run(
        command, capture_output=True, text=True, encoding="utf-8", errors="replace", check=False,
    )
    return completed.stdout, completed.stderr, completed.returncode


def combine_raw_streams(stdout: str, stderr: str) -> str:
    raw = stdout
    if stderr:
        if raw and not raw.endswith("\n"):
            raw += "\n"
        raw += stderr
    return raw


def parse_lizard_csv(csv_text: str, *, with_ens: bool = False) -> pd.DataFrame:
    if not csv_text.strip():
        columns = LIZARD_RAW_COLUMNS + (["max_nested_structures"] if with_ens else [])
        return pd.DataFrame(columns=columns)
    columns = LIZARD_RAW_COLUMNS + (["max_nested_structures"] if with_ens else [])
    rows = list(csv.reader(io.StringIO(csv_text.strip())))
    if rows and rows[0] and rows[0][0].lower() in {"nloc", "ncss"}:
        rows = rows[1:]
    parsed = [dict(zip(columns, row + [""] * (len(columns) - len(row)))) for row in rows]
    frame = pd.DataFrame(parsed)
    numeric_cols = [
        "nloc", "ccn", "token_count", "parameter_count", "length", "start_line", "end_line",
    ]
    if with_ens:
        numeric_cols.append("max_nested_structures")
    for col in numeric_cols:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    return frame


def to_lizard_output_csv(lizard_df: pd.DataFrame) -> pd.DataFrame:
    if lizard_df.empty:
        return pd.DataFrame(columns=LIZARD_OUTPUT_COLUMNS)
    return pd.DataFrame({
        "NLOC": lizard_df["nloc"],
        "CCN": lizard_df["ccn"],
        "token": lizard_df["token_count"],
        "PARAM": lizard_df["parameter_count"],
        "length": lizard_df["length"],
        "location": lizard_df["location"].astype(str).str.strip('"'),
        "file": lizard_df["file"].astype(str).str.strip('"'),
        "function": lizard_df["function"].astype(str).str.strip('"'),
    })


def to_lizard_metrics(lizard_df: pd.DataFrame) -> pd.DataFrame:
    if lizard_df.empty:
        return pd.DataFrame(columns=LIZARD_METRICS_COLUMNS)
    classes, methods = [], []
    for function_name in lizard_df["function"].astype(str):
        class_name, method_name = split_java_symbol(function_name)
        classes.append(class_name)
        methods.append(method_name)
    return pd.DataFrame({
        "file": lizard_df["file"].astype(str).str.strip('"'),
        "class": classes,
        "method": methods,
        "nloc": lizard_df["nloc"],
        "cyclomatic_complexity": lizard_df["ccn"],
        "token_count": lizard_df["token_count"],
        "parameter_count": lizard_df["parameter_count"],
        "function_length": lizard_df["length"],
        "start_line": lizard_df["start_line"],
        "end_line": lizard_df["end_line"],
    })


def build_long_parameter_list(metrics_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, record in metrics_df.iterrows():
        param_count = int(record.get("parameter_count", 0) or 0)
        status = "Long Parameter List" if param_count > LONG_PARAMETER_THRESHOLD else "OK"
        rows.append({
            "file": record.get("file", ""),
            "class": record.get("class", ""),
            "method": record.get("method", ""),
            "parameter_count": param_count,
            "status": status,
        })
    return pd.DataFrame(rows, columns=LONG_PARAMETER_LIST_COLUMNS)


def compute_max_nesting_depth(ens_df: pd.DataFrame) -> int:
    if ens_df.empty or "max_nested_structures" not in ens_df.columns:
        return 0
    values = pd.to_numeric(ens_df["max_nested_structures"], errors="coerce").dropna()
    return int(values.max()) if not values.empty else 0


def compute_complexity_summary(metrics_df: pd.DataFrame, max_nesting_depth: int) -> pd.DataFrame:
    ccn_values = pd.to_numeric(metrics_df["cyclomatic_complexity"], errors="coerce").dropna()
    param_values = pd.to_numeric(metrics_df["parameter_count"], errors="coerce").dropna()
    length_values = pd.to_numeric(metrics_df["function_length"], errors="coerce").dropna()
    return pd.DataFrame([
        {"metric_name": "Cyclomatic_Complexity", "metric_value": int(ccn_values.max()) if not ccn_values.empty else 0},
        {"metric_name": "Parameter_Count", "metric_value": int(param_values.max()) if not param_values.empty else 0},
        {"metric_name": "Function_Length", "metric_value": int(length_values.max()) if not length_values.empty else 0},
        {"metric_name": "Maximum_Nesting_Depth", "metric_value": max_nesting_depth},
    ])


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW LIZARD OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


## Section 4 — Repository Setup


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL,
        repo_url=REPO_URL,
        local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH,
        if_clone_exists=IF_CLONE_EXISTS,
        logger=logger,
        clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f'Repository setup failed: {exc}')
    raise

JAVA_FILES = discover_java_files(REPO_PATH)
if not JAVA_FILES:
    logger.error('No Java source files found in repository.', file=str(REPO_PATH))
    raise FileNotFoundError('No Java source files found.')

REPO_STATS = compute_repository_stats(REPO_PATH, JAVA_FILES)
logger.info(f'Repository ready at: {REPO_PATH}')
print(f"Repository: {REPO_STATS['repository_name']}")
print(f"Size (Java files): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Directories: {REPO_STATS['directory_count']:,}")
print(f"Java files: {REPO_STATS['java_file_count']:,}")


## Section 5 — Discover Java Files


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'java_files_inventory.csv'
save_java_inventory(JAVA_FILES, INVENTORY_CSV)

print(f'Total Java Files Found: {len(JAVA_FILES)}')
print(f'Saved inventory to: {INVENTORY_CSV}')


## Section 6 — Execute Lizard

Run `lizard -l java --csv` and nesting analysis with `-ENS` (equivalent to `-E NS`).

Preserve stdout and stderr exactly as emitted.


In [ ]:
LIZARD_CONSOLE_CHUNKS: list[str] = []
LIZARD_CSV = ''
LIZARD_CSV_ENS = ''

for label, csv_output, ens in [
    ('text', False, False),
    ('csv', True, False),
    ('csv_ens', True, True),
]:
    command = build_lizard_command(REPO_PATH, csv_output=csv_output, ens=ens)
    stdout, stderr, code = run_lizard_command(command, logger)
    LIZARD_CONSOLE_CHUNKS.append(f'===== lizard {label} =====\n' + combine_raw_streams(stdout, stderr))
    if csv_output and ens:
        LIZARD_CSV_ENS = stdout
    elif csv_output:
        LIZARD_CSV = stdout
    if code not in (0, 1):
        logger.error(f'Lizard {label} run exited with code {code}', file=label)

logger.info('Lizard execution complete.')


## Section 7 — Raw Output Extraction

Save raw console output and structured CSV files.


In [ ]:
CONSOLE_PATH = OUTPUT_PATH / 'lizard_raw_console_output.txt'
LIZARD_OUTPUT_CSV = OUTPUT_PATH / 'lizard_output.csv'

CONSOLE_PATH.write_text('\n'.join(LIZARD_CONSOLE_CHUNKS), encoding='utf-8')

BASE_DF = parse_lizard_csv(LIZARD_CSV, with_ens=False)
ENS_DF = parse_lizard_csv(LIZARD_CSV_ENS, with_ens=True)
OUTPUT_DF = to_lizard_output_csv(BASE_DF)
OUTPUT_DF.to_csv(LIZARD_OUTPUT_CSV, index=False)

logger.info(f'Saved raw console output and Lizard CSV ({len(OUTPUT_DF)} methods).')
preview_raw_output(CONSOLE_PATH.read_text(encoding='utf-8'), RAW_OUTPUT_PREVIEW_LINES, CONSOLE_PATH)


## Section 8 — Parse Lizard Output

Generate per-method metrics with class and method names.


In [ ]:
METRICS_DF = to_lizard_metrics(BASE_DF)
METRICS_CSV = OUTPUT_PATH / 'lizard_metrics.csv'
METRICS_DF.to_csv(METRICS_CSV, index=False)

logger.info(f'Parsed Lizard metrics rows={len(METRICS_DF)}')
display(METRICS_DF.head())


## Section 9 — Parameter Count

**Direct metric** (emitted by Lizard via the `PARAM` column):

```text
Parameter_Count = PARAM
```


In [ ]:
PARAM_VALUES = pd.to_numeric(METRICS_DF['parameter_count'], errors='coerce').dropna()
MAX_PARAM = int(PARAM_VALUES.max()) if not PARAM_VALUES.empty else 0
AVG_PARAM = round(float(PARAM_VALUES.mean()), 4) if not PARAM_VALUES.empty else 0.0

PARAM_SUMMARY_CSV = OUTPUT_PATH / 'parameter_count_summary.csv'
pd.DataFrame([{'metric_name': 'Parameter_Count', 'metric_value': MAX_PARAM}]).to_csv(PARAM_SUMMARY_CSV, index=False)

logger.info(f'Parameter Count (max PARAM)={MAX_PARAM} (directly from Lizard)')
display(pd.read_csv(PARAM_SUMMARY_CSV))


## Section 10 — Long Parameter List Detection

Flag methods where `Parameter_Count > 5`.


In [ ]:
LONG_PARAM_DF = build_long_parameter_list(METRICS_DF)
LONG_PARAM_CSV = OUTPUT_PATH / 'long_parameter_list.csv'
LONG_PARAM_DF.to_csv(LONG_PARAM_CSV, index=False)

LONG_PARAM_COUNT = int((LONG_PARAM_DF['status'] == 'Long Parameter List').sum())
logger.info(f'Long parameter list count={LONG_PARAM_COUNT}')
display(LONG_PARAM_DF)


## Section 11 — Complexity Summary


In [ ]:
MAX_NESTING = compute_max_nesting_depth(ENS_DF)
COMPLEXITY_DF = compute_complexity_summary(METRICS_DF, MAX_NESTING)
COMPLEXITY_CSV = OUTPUT_PATH / 'complexity_summary.csv'
COMPLEXITY_DF.to_csv(COMPLEXITY_CSV, index=False)

CCN_VALUES = pd.to_numeric(METRICS_DF['cyclomatic_complexity'], errors='coerce').dropna()
AVG_CCN = round(float(CCN_VALUES.mean()), 4) if not CCN_VALUES.empty else 0.0

logger.info(f'Saved complexity summary: {COMPLEXITY_CSV}')
display(COMPLEXITY_DF)


## Section 12 — Summary Dashboard


In [ ]:
summary_df = pd.DataFrame([
    {'Metric': 'Total Java Files', 'Value': len(JAVA_FILES)},
    {'Metric': 'Total Methods', 'Value': len(METRICS_DF)},
    {'Metric': 'Average Parameter Count', 'Value': AVG_PARAM},
    {'Metric': 'Maximum Parameter Count', 'Value': MAX_PARAM},
    {'Metric': 'Long Parameter List Count', 'Value': LONG_PARAM_COUNT},
    {'Metric': 'Average Cyclomatic Complexity', 'Value': AVG_CCN},
    {'Metric': 'Maximum Nesting Depth', 'Value': MAX_NESTING},
])
display(summary_df)

deliverables = [
    CONSOLE_PATH, LIZARD_OUTPUT_CSV, METRICS_CSV, PARAM_SUMMARY_CSV,
    LONG_PARAM_CSV, COMPLEXITY_CSV, INVENTORY_CSV, ERROR_LOG_PATH,
]
print('\nDeliverables:')
for path in deliverables:
    print(f"  [{'OK' if path.exists() else 'MISSING'}] {path}")


## Section 13 — Error Handling


In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding='utf-8'))
else:
    print('No errors logged.')


## Section 14 — Deliverables

```text
outputs/
├── lizard_raw_console_output.txt
├── lizard_output.csv
├── lizard_metrics.csv
├── parameter_count_summary.csv
├── long_parameter_list.csv
├── complexity_summary.csv
├── java_files_inventory.csv
└── error_log.txt
```
